# 第2部分：IV值计算（信息价值）

**目的：** 衡量每个特征对"好坏客户"的区分能力

## 通俗理解

IV就像考试成绩——分数越高，说明这个特征越能区分好人和坏人：

| IV值 | 含义 | 举例 |
|------|------|------|
| < 0.02 | 没用 | 比如客户的星座 |
| 0.02~0.1 | 弱 | 比如注册渠道 |
| 0.1~0.3 | 中等 | 比如历史逾期次数 |
| 0.3~0.5 | 强 | 比如信用评分 |
| > 0.5 | 过强(可能有问题) | 可能信息泄漏 |

In [ ]:
import numpy as np
import pandas as pd
from optbinning import OptimalBinning

## 步骤1：数据预处理

在计算IV之前，需要对数据做基本清洗

In [ ]:
def preprocess_for_iv(df, label_col='dob4_ever10_flg'):
    """
    IV计算前的预处理

    做两件事：
    1. 去掉全是空值的列（没数据算不了IV）
    2. 去掉只有一个值的列（所有人都一样，没区分力）
    """
    # 只保留数值型特征
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # 去掉标签列本身
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)

    # 过滤：空值率<100% 且 不同值>1个
    valid_cols = []
    for col in numeric_cols:
        if df[col].notna().sum() > 0 and df[col].nunique() > 1:
            valid_cols.append(col)

    print(f'原始特征数: {len(numeric_cols)}, 有效特征数: {len(valid_cols)}')
    return valid_cols

## 步骤2：计算单个特征的IV

### WOE和IV的关系

```
WOE = ln(好人占比 / 坏人占比)   ← 每个箱一个值
IV  = Σ (好人占比 - 坏人占比) × WOE   ← 所有箱的加总
```

### 为什么用OptimalBinning分箱？

手动分箱（等频/等距）可能切在不好的位置，OptimalBinning自动找最佳切分点。

In [ ]:
def calculate_iv(series, target, var_name='feature'):
    """
    计算单个特征的IV值

    参数：
        series: 特征数据（一列）
        target: 标签（0=好人, 1=坏人）
        var_name: 特征名（用于日志）

    返回：
        float: IV值
    """
    try:
        # 去掉空值
        mask = series.notna() & target.notna()
        x = series[mask].values
        y = target[mask].values.astype(int)

        if len(x) == 0 or len(np.unique(y)) < 2:
            return 0.0

        # OptimalBinning自动找最佳分箱
        optb = OptimalBinning(
            name=var_name,
            dtype='numerical',
            solver='cp',           # 约束规划求解器
            max_n_bins=5,          # 最多5箱
            min_bin_size=0.05      # 每箱至少5%样本
        )
        optb.fit(x, y)

        # 从分箱结果中提取IV
        table = optb.binning_table.build()
        iv = table['IV'].iloc[:-2].sum()  # 去掉Missing和Totals行

        return iv

    except Exception as e:
        return 0.0

## 步骤3：并行批量计算IV

特征有几百个，一个一个算太慢，用多进程并行加速

In [ ]:
from multiprocessing import Pool, cpu_count
from functools import partial


def compute_iv_fast(df, feature_list, label_col='dob4_ever10_flg', n_jobs=None):
    """
    并行计算所有特征的IV

    参数：
        df: 数据
        feature_list: 要计算IV的特征列表
        label_col: 标签列名
        n_jobs: 并行进程数（默认=CPU核数）

    返回：
        DataFrame: 特征名 + IV值，按IV降序排列
    """
    if n_jobs is None:
        n_jobs = cpu_count()

    target = df[label_col]

    # 构造参数列表
    args_list = [(df[col], target, col) for col in feature_list]

    # 多进程并行计算
    with Pool(n_jobs) as pool:
        iv_values = pool.starmap(calculate_iv, args_list)

    # 整理结果
    result = pd.DataFrame({
        'variable': feature_list,
        'iv': iv_values
    }).sort_values('iv', ascending=False)

    print(f'IV>0.02的特征数: {(result["iv"] > 0.02).sum()}')
    print(f'IV>0.1的特征数: {(result["iv"] > 0.1).sum()}')

    return result

## 实际使用

In [ ]:
# 1. 预处理
valid_features = preprocess_for_iv(df)

# 2. 并行计算IV
iv_result = compute_iv_fast(df, valid_features)

# 3. 筛选IV>0.02的特征进入下一步
selected_features = iv_result[iv_result['iv'] > 0.02]['variable'].tolist()
print(f'\n选出 {len(selected_features)} 个有效特征进入建模')

# 查看Top 10特征
iv_result.head(10)

---
### 面试考点

| 问题 | 答案 |
|------|------|
| IV>0.5为什么可能有问题？ | 可能存在信息泄漏（用了未来数据） |
| 为什么用OptimalBinning而不是等频分箱？ | 自动找最优切分点，IV计算更准确 |
| WOE编码有什么好处？ | 把分类变量变成连续值，且自带单调性 |
| 为什么加Laplace平滑(+1e-6)？ | 防止某箱好人或坏人为0导致log(0) |